## EU REED

#### 1) Import packages and set filepath

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tarfile
from pathlib import Path
from reed import utils
from collections import defaultdict
from functools import reduce

In [2]:
config = utils.Config()

In [3]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"

#### 2) Import foreground database 

In [4]:
all_records = []

for company_folder in monitoring_data_path.iterdir():
    if not company_folder.is_dir():
        continue

    parts = [p.strip() for p in company_folder.name.split(" - ", 1)]
    if len(parts) != 2:
        print("Advarsel – uventet mappenavn:", company_folder.name)
        continue

    company, machine = parts

    # Find .tar.gz
    tar_files = list(company_folder.glob("*.tar.gz"))
    # Hvis dine filer evt. ligger i undermapper:
    # tar_files = list(company_folder.rglob("*.tar.gz"))

    if not tar_files:
        print(f"Ingen .tar.gz-filer i {company_folder.name}")
        continue

    for tar_path in tar_files:
        stem = tar_path.name.replace(".tar.gz", "")  # fx OF10001_G_BQC_S8CF2G_7N4ZJ8
        try:
            event_id, measurement_type = stem.rsplit("_", 1)
        except ValueError:
            event_id, measurement_type = stem, None

        with tarfile.open(tar_path, "r:gz") as tf:  # ← vigtigt: r:gz
            csv_members = [m for m in tf.getmembers()
                           if m.name.lower().endswith(".csv")]

            if not csv_members:
                print(f"Ingen CSV fundet i {tar_path.name}")
                continue

            csv_member = csv_members[0]
            with tf.extractfile(csv_member) as f:
                df = pd.read_csv(f)

        all_records.append({
            "company": company,
            "maskinenavn": machine,
            "event_id": event_id,
            "measurement_type": measurement_type,
            "tar_path": tar_path,
            "csv_name": csv_member.name,
            "data": df,
        })

print("\nTotal amount of excelsheets:", len(all_records))

# === group company and event ===
events = defaultdict(list)
for rec in all_records:
    events[(rec["company"], rec["event_id"])].append(rec)

print("Events:", len(events))



Total amount of excelsheets: 61
Events: 16


#### 3) Explore processes in the databases

In [13]:
def merge_records(records):
    """
    Creating one dataFrame for each event.

    """
    dfs = []
    cols = ["timestamp", "Tool_Number", "Power_Active", "Energy_Total"]

    merged = None
    
    for i, r in enumerate(records):
        df = r["data"].copy()
        df = df[[c for c in cols if c in df.columns]].copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.floor('s')
        df = df.set_index("timestamp")

        if merged is None:
            merged = df
        else:
            merged = merged.join(df, how="outer")
            
    return merged

In [14]:
grouped_data = {}

for (virksomhed, event_id), records in events.items():
    merged_df = merge_records(records)

    merged_df["energy_consumption"] = merged_df["Power_Active"] / 3600

    # Gruppér efter tool_number
    grouped_df = (
        merged_df
        .groupby("Tool_Number", as_index=False)
        .agg({
            "energy_consumption": "sum",
            "Power_Active": "mean",
            "Energy_Total": lambda x: x.max() - x.min()
        })
    )

    grouped_data[(virksomhed, event_id)] = grouped_df

In [15]:
df = grouped_data[("GOIMEK", "OF10001_G_BQC_S8CF2G")]

In [ ]:
problems = []

for key, records in events.items():
    try:
        merged_df = merge_records(records)

        # Ekstra check: krævede kolonner skal findes
        for col in ["Power_Active", "Energy_Total", "Tool_Number"]:
            if col not in merged_df.columns:
                print(f"⚠️  Mangler kolonne '{col}' i event {key}")

        # Prøv også den næste operation i din pipeline:
        merged_df["energy_consumption"] = merged_df["Power_Active"] / 3600

        # Og groupby (uden at gemme resultatet)
        test_group = (
            merged_df
            .groupby("Tool_Number", as_index=False)
            .agg({
                "energy_consumption": "sum",
                "Power_Active": "mean",
                "Energy_Total": lambda x: x.max() - x.min()
            })
        )

        print(f"✔️  {key} OK ({len(merged_df)} rækker)")

    except Exception as e:
        print(f"❌  {key} FEJLER")
        print("    Fejl:", e)
        problems.append((key, e))


In [ ]:
print(grouped_data)